# Pulso Penonomé: el sismo de 2026 en Coclé y dónde ayudar primero

Análisis principal del proyecto del equipo RacoonCity. Sigue el flujo completo de un análisis de datos sobre las afectaciones sociales del terremoto del 9 de octubre de 2026 en Coclé (Panamá) y sobre la vulnerabilidad de los 16 corregimientos de Penonomé, con lo aprendido en las semanas 1 a 6 del módulo de Python.

[Proyecto](README.md) | [Estudio completo](documento/estudio-penonome.md) | [Anexo: técnicas por semana](notebooks/anexo-semanas/s01-listas-diccionarios-conjuntos.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## 1. La pregunta

El 9 de octubre de 2026 un sismo de magnitud 7.5 según SINAPROC (7.7 según el USGS), con epicentro en Azuero, dañó viviendas, escuelas y hospitales de Coclé. En Penonomé, la capital de la provincia (104,326 habitantes en 16 corregimientos, Censo 2023), se dañaron la catedral, la Gobernación, el mercado y el hospital.

El análisis responde tres preguntas con datos oficiales verificados, desde la salud pública:

1. **¿Cómo estaba Penonomé antes del sismo?** Población, pobreza multidimensional y carencias de las viviendas por corregimiento.
2. **¿Qué dejó el sismo en Coclé?** Viviendas evaluadas por nivel de daño, quiénes viven en ellas (edad, discapacidad, embarazo), escuelas, hospitales, albergues, ayuda alimentaria y réplicas.
3. **¿Dónde ayudar primero?** Un índice de prioridad por corregimiento de Penonomé.

SINAPROC publica el daño por provincia y el Censo y el PNUD publican la vulnerabilidad por corregimiento. Por eso el análisis mira Coclé para el daño y Penonomé para saber dónde vivía la gente más expuesta.

## 2. El flujo de análisis

```mermaid
flowchart LR
    A[Fuentes oficiales<br/>INEC, PNUD, SINAPROC] --> B[Carga<br/>csv y pandas]
    B --> C[Limpieza y verificación<br/>sumas de control]
    C --> D[Análisis<br/>antes, sismo, después]
    D --> E[Visualización<br/>gráficas]
    E --> F[Decisión<br/>dónde ayudar primero]
```

| Paso | Qué se hace | Lo aprendido que se usa |
|---|---|---|
| Carga | Leer las tablas oficiales | Módulos y paquetes (semana 5), `pandas.read_csv` (semana 5) |
| Limpieza | Unir tablas, tipos, valores faltantes, controles | Diccionarios y conjuntos (semana 1), funciones (semana 2), `merge` y datos ordenados (semana 6) |
| Análisis | Proporciones, cruces de daño y vulnerabilidad, cambios entre cortes | Funciones y lambda (semana 2), `groupby` y series temporales (semana 6) |
| Prioridad | Ordenar corregimientos por necesidad | Ordenamiento por fusión (semana 4), búsqueda binaria (semana 3) |
| Visualización | Gráficas para el estudio y la presentación | matplotlib (semanas 5 y 6) |

Las técnicas se explican una por una en el anexo (`notebooks/anexo-semanas/`, de `s01` a `s06`); aquí se usan para contar la historia de Penonomé.

## 3. Carga

Todo el código reutilizable vive en el paquete `penonome` (`src/penonome/`). El módulo `pulso` carga las tablas del análisis desde `datos/`: los datos oficiales de Penonomé (`datos/penonome/`: Censo 2023 y pobreza multidimensional del PNUD) y las tablas del sismo (`datos/analisis/`: evaluación de daños de SINAPROC, registros por corregimiento, réplicas y situación actual).

**Lo aprendido:** módulos y paquetes ([unidad 34](../../01-python/semana-05/clase-2-modulos-y-pandas/u34-modulos-de-python.ipynb)) y DataFrames ([unidad 36](../../01-python/semana-05/clase-2-modulos-y-pandas/u36-dataframes-de-pandas.ipynb)).

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from penonome import pulso
from penonome.pulso import AZUL, NARANJA, AGUA, GRIS, TINTA2, estilo, figura, guardar
from penonome.ordenamiento import fusion
from penonome.busqueda import bin_search

t = pulso.cargar_todo()
for nombre, tabla in t.items():
    print(f"{nombre:18} {tabla.shape[0]:>4} filas x {tabla.shape[1]:>2} columnas")

catalogo             16 filas x  5 columnas
poblacion            16 filas x 10 columnas
viviendas            16 filas x 23 columnas
viviendas_totales     2 filas x 23 columnas
ipm                  16 filas x  2 columnas
danos                 3 filas x  5 columnas
afectaciones         40 filas x 10 columnas
provincias           11 filas x  7 columnas
registros            16 filas x  6 columnas
replicas              7 filas x  3 columnas
poblacion_proy        6 filas x  6 columnas
situacion            11 filas x  9 columnas


## 4. Limpieza y verificación

Antes de analizar se comprueba que los datos cuadren: los corregimientos deben sumar el distrito y, en la evaluación de SINAPROC, los niveles de daño, los grupos de edad y las personas que necesitan atención especial deben sumar sus totales. Si una suma no da, el error está en los datos y no en el análisis.

**Lo aprendido:** diccionarios y conjuntos ([unidades 14 a 16](../../01-python/semana-01/clase-2-diccionarios-y-conjuntos/u16-conjuntos.ipynb)), funciones ([unidad 17](../../01-python/semana-02/clase-1-funciones-recursividad-lambda/u17-funciones.ipynb)) y `merge` de pandas ([unidad 37](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u37-ordenacion-de-datos.ipynb)).

In [3]:
corr = pulso.tabla_corregimientos(t)
distrito = t["viviendas_totales"].set_index("nivel").loc["distrito"]

# Conjuntos: ¿los 16 corregimientos aparecen en todas las tablas que se unieron?
nombres = {"catálogo": set(t["catalogo"]["corregimiento"]), "pobreza": set(t["ipm"]["corregimiento"]),
           "daños": set(t["registros"]["corregimiento"])}
faltan = {k: nombres["catálogo"] - v for k, v in nombres.items()}
print("Corregimientos sin pareja en alguna tabla:", {k: v for k, v in faltan.items() if v} or "ninguno")
print("Valores faltantes tras unir:", int(corr.isna().sum().sum()))

def tasa(parte, total):
    """Porcentaje con un decimal."""
    return round(parte / total * 100, 1)

verificar("los 16 corregimientos suman la población del distrito", int(corr["pob_2023"].sum()), 104326)
verificar("las viviendas de los corregimientos suman las del distrito", int(corr["viviendas"].sum()), int(distrito["viviendas"]))
verificar("la unión no pierde corregimientos", len(corr), 16)

# Evaluación de daños de SINAPROC (formato largo): cada desglose debe sumar su total
af = t["afectaciones"]

def valor(categoria, subcategoria=None, desagregacion=None):
    """Suma la columna valor de las filas que cumplen las condiciones dadas."""
    filas = af[af["categoria"] == categoria]
    if subcategoria:
        filas = filas[filas["subcategoria"] == subcategoria]
    if desagregacion:
        filas = filas[filas["desagregacion"] == desagregacion]
    return int(filas["valor"].sum())

evaluadas = valor("Cobertura", "Viviendas evaluadas")
personas = valor("Cobertura", "Personas en las viviendas evaluadas")
verificar("los niveles de daño suman las viviendas evaluadas", valor("Viviendas por nivel de daño"), evaluadas)
verificar("los grupos de edad y sexo suman las personas evaluadas", valor("Edad y sexo"), personas)
for grupo in ("Personas con discapacidad", "Mujeres embarazadas"):
    verificar(f"{grupo.lower()}: el desglose por nivel de daño suma el total",
              valor("Atención especial por nivel de daño", desagregacion=grupo), valor("Atención especial", grupo))

Corregimientos sin pareja en alguna tabla: ninguno
Valores faltantes tras unir: 0
Correcto: los 16 corregimientos suman la población del distrito
Correcto: las viviendas de los corregimientos suman las del distrito
Correcto: la unión no pierde corregimientos
Correcto: los niveles de daño suman las viviendas evaluadas
Correcto: los grupos de edad y sexo suman las personas evaluadas
Correcto: personas con discapacidad: el desglose por nivel de daño suma el total
Correcto: mujeres embarazadas: el desglose por nivel de daño suma el total


## 5. ¿Cómo estaba Penonomé antes del sismo?

### 5.1 ¿Dónde vive la gente y dónde está la pobreza?

La población se concentra en el sur urbano (la cabecera, El Coco y Cañaveral), mientras la pobreza se concentra en el norte rural y montañoso. Uno de los 16 corregimientos se llama Coclé, como la provincia; en las gráficas aparece como Coclé (corregimiento). El gráfico ordena los corregimientos por población y los colorea por su pobreza multidimensional de 2023.

**Lo aprendido:** ordenamiento por fusión ([unidad 28](../../01-python/semana-04/clase-1-algoritmos-de-ordenamiento/u28-ordenamiento-por-fusion.ipynb)) para ordenar los corregimientos.

In [4]:
filas = corr.to_dict("records")
ordenados, comparaciones = fusion(filas, clave=lambda c: c["pob_2023"])
verificar("la fusión ordena igual que sorted()", [c["corregimiento"] for c in ordenados],
          [c["corregimiento"] for c in sorted(filas, key=lambda c: c["pob_2023"])])

fig, ax = figura(10, 5.8)
colores = [NARANJA if c["ipm"] >= 45 else AZUL for c in ordenados]
barras = ax.barh([c["corregimiento"] for c in ordenados], [c["pob_2023"] for c in ordenados], color=colores, height=0.7)
for b, c in zip(barras, ordenados):
    ax.text(b.get_width() + 250, b.get_y() + b.get_height() / 2, f"{c['pob_2023']:,}  (pobreza {c['ipm']:.0f} %)",
            va="center", fontsize=8.5, color=TINTA2)
ax.set_xlim(0, 33000)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=AZUL), plt.Rectangle((0, 0), 1, 1, color=NARANJA)],
          labels=["Pobreza multidimensional menor a 45 %", "45 % o más"], frameon=False, loc="lower right")
estilo(ax, "Penonomé: la gente vive en el sur y la pobreza está en el norte",
       "INEC, Censo 2023, Cuadro 4; PNUD/MEF, IPM por corregimiento 2023.", xlabel="Habitantes en 2023", rejilla="x")
guardar(fig, "pulso-01-poblacion-pobreza")
plt.show()

sur = corr[corr["ipm"] < 45]
print(f"Sur (pobreza < 45 %): {len(sur)} corregimientos, {sur['pob_2023'].sum():,} habitantes "
      f"({tasa(sur['pob_2023'].sum(), 104326)} % del distrito)")

Correcto: la fusión ordena igual que sorted()


Sur (pobreza < 45 %): 6 corregimientos, 64,036 habitantes (61.4 % del distrito)


/tmp/ipykernel_1022084/725270340.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** seis corregimientos del sur reúnen al 61 % de la población con pobreza baja; los diez del norte tienen pobreza de 45 % a 86 %.

### 5.2 ¿Qué carencias tenían las viviendas?

Con el Censo 2023 se comparan las carencias de las viviendas del distrito, de Coclé y de la República.

**Lo aprendido:** funciones que reciben una tabla y devuelven porcentajes ([unidad 17](../../01-python/semana-02/clase-1-funciones-recursividad-lambda/u17-funciones.ipynb)) y `map` con `lambda` ([unidad 19](../../01-python/semana-02/clase-1-funciones-recursividad-lambda/u19-lambda-filter-map-reduce.ipynb)).

In [5]:
# Fila de la República en el Censo 2023, Cuadro 1 (no está en el paquete por corregimiento)
republica = {"viviendas": 1201809, "piso_tierra": 65379, "sin_agua_potable": 60679, "sin_luz": 58032,
             "cocinan_lena": 72179, "sin_internet": 363721}
areas = {"República": republica, "Provincia de Coclé": t["viviendas_totales"].set_index("nivel").loc["provincia"],
         "Distrito de Penonomé": distrito}
items = {"piso_tierra": "Piso de tierra", "sin_agua_potable": "Sin agua potable", "sin_luz": "Sin luz eléctrica",
         "cocinan_lena": "Cocinan con leña", "sin_internet": "Sin internet"}
carencias = pd.DataFrame({a: list(map(lambda k: tasa(v[k], v["viviendas"]), items)) for a, v in areas.items()},
                         index=list(items.values()))
print(carencias)

fig, ax = figura(10, 5.2)
y = range(len(carencias))
for i, (area, color) in enumerate(zip(carencias.columns, (GRIS, AZUL, NARANJA))):
    pos = [k + (i - 1) * 0.27 for k in y]
    ax.barh(pos, carencias[area], height=0.25, color=color, label=area)
    for p, v in zip(pos, carencias[area]):
        ax.text(v + 0.4, p, f"{v:.1f}", va="center", fontsize=8.5, color=TINTA2)
ax.set_yticks(list(y), carencias.index)
ax.invert_yaxis()
ax.set_xlim(0, 52)
ax.legend(frameon=False, loc="upper right")
estilo(ax, "El distrito arrastra más carencias que el país", "INEC, Censo 2023, Cuadro 1. Porcentajes calculados.",
       xlabel="% de viviendas particulares ocupadas", rejilla="x")
guardar(fig, "pulso-02-carencias")
plt.show()

verificar("en el distrito 41.9 % de las viviendas no tiene internet", carencias.loc["Sin internet", "Distrito de Penonomé"], 41.9)
verificar("en el país 30.3 % no tiene internet", carencias.loc["Sin internet", "República"], 30.3)

                   República  Provincia de Coclé  Distrito de Penonomé
Piso de tierra           5.4                 5.8                   9.3
Sin agua potable         5.0                 3.8                   5.9
Sin luz eléctrica        4.8                 7.0                   7.7
Cocinan con leña         6.0                10.8                  15.7
Sin internet            30.3                44.0                  41.9
Correcto: en el distrito 41.9 % de las viviendas no tiene internet
Correcto: en el país 30.3 % no tiene internet


/tmp/ipykernel_1022084/2158287081.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** en las cinco carencias del gráfico el distrito está peor que el país; la brecha más grande es internet (41.9 % frente a 30.3 %).

## 6. ¿Qué dejó el sismo en Coclé?

### 6.1 ¿Cómo crecieron las cifras oficiales?

Los primeros comunicados de SINAPROC sumaban lo evaluado hasta cada corte: en cuatro días las viviendas afectadas en Coclé pasaron de 149 a 1,324. La evaluación de daños siguió hasta el 15 de noviembre; su balance final se analiza en 6.2.

**Lo aprendido:** variación entre cortes con `pct_change` ([unidad 38](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u38-series-temporales.ipynb)).

In [6]:
danos = t["danos"].assign(fecha=lambda x: pd.to_datetime(x["fecha_hora"]))
danos["crecimiento_pct"] = (danos["viviendas_afectadas"].pct_change() * 100).round(0)
print(danos[["corte", "fecha", "viviendas_afectadas", "crecimiento_pct"]])

fig, ax = figura(10, 4.8)
etiquetas = [f"{c}\n{f:%d-%b}".replace("Oct", "oct") for c, f in zip(danos["corte"], danos["fecha"])]
barras = ax.bar(etiquetas, danos["viviendas_afectadas"], color=[AZUL, AZUL, NARANJA], width=0.55)
for b in barras:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 25, f"{int(b.get_height()):,}", ha="center", fontsize=12,
            fontweight="bold", color=TINTA2)
ax.set_ylim(0, 1550)
estilo(ax, "Viviendas afectadas en Coclé: de 149 a 1,324 en cuatro días",
       "SINAPROC, comunicados #5, #7 y #12 (balances preliminares).", ylabel="Viviendas afectadas")
guardar(fig, "pulso-05-danos")
plt.show()
verificar("el último balance es 8.9 veces el primero", round(danos["viviendas_afectadas"].iloc[-1] / danos["viviendas_afectadas"].iloc[0], 1), 8.9)

                         corte               fecha  viviendas_afectadas  \
0                Comunicado #5 2026-10-10 18:30:00                  149   
1                Comunicado #7 2026-10-11 12:00:00                  229   
2  Comunicado #12 (preliminar) 2026-10-14 12:00:00                 1324   

   crecimiento_pct  
0              NaN  
1             54.0  
2            478.0  


Correcto: el último balance es 8.9 veces el primero


/tmp/ipykernel_1022084/3709903541.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** en cinco días las cifras se multiplicaron por casi nueve. Eran balances preliminares que crecían a medida que llegaban evaluaciones, así que siempre hay que decir de qué corte es una cifra.

### 6.2 ¿Cómo quedaron las viviendas evaluadas?

La evaluación de daños y análisis de necesidades (EDAN) de SINAPROC cerró con cifras finales al 15 de noviembre de 2026. Son las viviendas que se inspeccionaron, no todas las de la provincia, así que los porcentajes se calculan sobre ellas. «Daño total» es la categoría más grave de la evaluación; no equivale a colapso.

**Lo aprendido:** filtrar un DataFrame, `set_index` y `reindex` para fijar un orden ([unidad 36](../../01-python/semana-05/clase-2-modulos-y-pandas/u36-dataframes-de-pandas.ipynb)).

In [7]:
orden_dano = ["Total", "Moderado", "Parcial", "Leve", "Menor"]
niveles = af[af["categoria"] == "Viviendas por nivel de daño"].set_index("subcategoria")["valor"].reindex(orden_dano)
pct = (niveles / evaluadas * 100).round(1)
print(pd.DataFrame({"viviendas": niveles, "% de las evaluadas": pct}))
grave = int(niveles[["Total", "Moderado"]].sum())
print(f"Daño total o moderado: {grave:,} de {evaluadas:,} viviendas ({tasa(grave, evaluadas)} %)")

fig, ax = figura(10, 4.8)
barras = ax.bar(orden_dano, niveles.values, color=[NARANJA, NARANJA, AZUL, GRIS, GRIS], width=0.6)
for b, v, p in zip(barras, niveles.values, pct.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 30, f"{v:,}\n{p:.1f} %", ha="center", fontsize=10, color=TINTA2)
ax.set_ylim(0, niveles.max() * 1.22)
estilo(ax, f"Coclé: {grave:,} de {evaluadas:,} viviendas evaluadas con daño total o moderado",
       "SINAPROC, EDAN del 9-oct al 15-nov-2026 (cifras finales), plataforma oficial de afectaciones del sismo.",
       xlabel="Nivel de daño", ylabel="Viviendas evaluadas")
guardar(fig, "pulso-03-nivel-dano")
plt.show()

# ¿Cómo se compara Coclé con las provincias más afectadas?
prov = t["provincias"].sort_values("viviendas_evaluadas", ascending=False)
prov["personas_por_vivienda"] = (prov["personas_en_viviendas_evaluadas"] / prov["viviendas_evaluadas"]).round(1)
print(prov.head(4)[["provincia", "viviendas_evaluadas", "personas_en_viviendas_evaluadas", "personas_por_vivienda"]]
      .to_string(index=False))
verificar("Coclé es la tercera provincia con más viviendas evaluadas", prov["provincia"].tolist().index("Coclé") + 1, 3)
verificar("561 viviendas con daño total o moderado", grave, 561)

              viviendas  % de las evaluadas
subcategoria                               
Total               414                16.4
Moderado            147                 5.8
Parcial            1750                69.1
Leve                145                 5.7
Menor                75                 3.0
Daño total o moderado: 561 de 2,531 viviendas (22.2 %)


 provincia  viviendas_evaluadas  personas_en_viviendas_evaluadas  personas_por_vivienda
Los Santos                11262                            32034                    2.8
   Herrera                 6572                            20300                    3.1
     Coclé                 2531                            10199                    4.0
  Veraguas                  816                             3175                    3.9
Correcto: Coclé es la tercera provincia con más viviendas evaluadas
Correcto: 561 viviendas con daño total o moderado


/tmp/ipykernel_1022084/2576897047.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** 561 de las 2,531 viviendas evaluadas (22.2 %) quedaron con daño total o moderado, y la mayoría, 1,750 (69.1 %), con daño parcial. Coclé es la tercera provincia con más viviendas evaluadas, detrás de Los Santos y Herrera, y en cada vivienda evaluada viven en promedio 4.0 personas.

### 6.3 ¿Quiénes viven en las viviendas dañadas?

La EDAN registra quién vive en cada vivienda evaluada, sin datos de las familias. Para la salud pública importan los grupos que necesitan atención especial: la niñez, los adultos mayores, las personas con discapacidad y las mujeres embarazadas, sobre todo si su casa quedó con daño grave.

**Lo aprendido:** `pivot_table` para pasar de formato largo a una tabla de doble entrada ([unidad 37](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u37-ordenacion-de-datos.ipynb)).

In [8]:
orden_edad = ["Recién nacidos", "Niñez", "Adolescencia", "Adultos", "Adultos mayores"]
edad = (af[af["categoria"] == "Edad y sexo"]
        .pivot_table(index="subcategoria", columns="desagregacion", values="valor", aggfunc="sum").reindex(orden_edad))
edad["Total"] = edad.sum(axis=1)
edad["% de las personas"] = (edad["Total"] / personas * 100).round(1)
print(edad, end="\n\n")

vul = (af[af["categoria"] == "Atención especial por nivel de daño"]
       .pivot_table(index="subcategoria", columns="desagregacion", values="valor", aggfunc="sum").reindex(orden_dano))
print(vul)
graves = vul.loc[["Total", "Moderado"]].sum()
print(f"\nEn viviendas con daño total o moderado: {graves['Personas con discapacidad']} personas con discapacidad "
      f"y {graves['Mujeres embarazadas']} mujeres embarazadas")

fig, ax = figura(10, 4.8)
y = range(len(orden_dano))
disc = vul["Personas con discapacidad"]
barras = ax.barh(list(y), disc.values, color=[NARANJA, NARANJA, AZUL, GRIS, GRIS], height=0.6)
for b, nivel in zip(barras, orden_dano):
    ax.text(b.get_width() + 4, b.get_y() + b.get_height() / 2,
            f"{int(disc[nivel])} con discapacidad, {int(vul.loc[nivel, 'Mujeres embarazadas'])} "
            f"{'embarazada' if vul.loc[nivel, 'Mujeres embarazadas'] == 1 else 'embarazadas'}",
            va="center", fontsize=9, color=TINTA2)
ax.set_yticks(list(y), [f"Daño {n.lower()}" for n in orden_dano])
ax.invert_yaxis()
ax.set_xlim(0, disc.max() * 1.7)
estilo(ax, f"{int(graves['Personas con discapacidad'])} con discapacidad y {int(graves['Mujeres embarazadas'])} "
           f"embarazadas viven en casas con daño grave",
       "SINAPROC, EDAN del 9-oct al 15-nov-2026, Coclé. Daño grave = total o moderado.",
       xlabel="Personas con discapacidad en las viviendas evaluadas", rejilla="x")
guardar(fig, "pulso-04-vulnerables")
plt.show()

verificar("91 personas con discapacidad en viviendas con daño total o moderado", int(graves["Personas con discapacidad"]), 91)
verificar("9 mujeres embarazadas en viviendas con daño total o moderado", int(graves["Mujeres embarazadas"]), 9)

desagregacion    Hombres  Mujeres  Total  % de las personas
subcategoria                                               
Recién nacidos       172      151    323                3.2
Niñez               1090      897   1987               19.5
Adolescencia         455      458    913                9.0
Adultos             2521     2720   5241               51.4
Adultos mayores      873      862   1735               17.0

desagregacion  Mujeres embarazadas  Personas con discapacidad
subcategoria                                                 
Total                            7                         77
Moderado                         2                         14
Parcial                         11                        314
Leve                             1                         13
Menor                            0                         13

En viviendas con daño total o moderado: 91 personas con discapacidad y 9 mujeres embarazadas


Correcto: 91 personas con discapacidad en viviendas con daño total o moderado
Correcto: 9 mujeres embarazadas en viviendas con daño total o moderado


/tmp/ipykernel_1022084/3098952797.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** 91 personas con discapacidad y 9 mujeres embarazadas viven en viviendas con daño total o moderado: son las primeras que necesitan seguimiento médico y un lugar seguro. Entre todas las personas de las viviendas evaluadas, 22.6 % son recién nacidos o niños y 17.0 % son adultos mayores.

### 6.4 ¿Qué servicios quedaron afectados?

Además de las viviendas, el COE Nacional registra en Coclé escuelas, hospitales y edificios públicos afectados, y la ayuda alimentaria entregada. La plataforma actualiza este bloque a diario; los valores son los de la consulta del 25 de noviembre de 2026.

**Lo aprendido:** una Serie como diccionario con etiquetas: se consulta cada valor por su nombre ([unidad 14](../../01-python/semana-01/clase-2-diccionarios-y-conjuntos/u14-diccionarios-crud.ipynb) y [unidad 36](../../01-python/semana-05/clase-2-modulos-y-pandas/u36-dataframes-de-pandas.ipynb)).

In [9]:
otras = af[af["categoria"] == "Otras afectaciones"].set_index("subcategoria")["valor"]
print(otras.to_string(), end="\n\n")
por_comunidad = otras["Bolsas de alimentos del IMA"] / otras["Comunidades atendidas por el IMA"]
print(f"Bolsas de alimentos del IMA por comunidad atendida: {por_comunidad:.1f}")
verificar("no quedan albergues activos en Coclé", int(otras["Albergues activos"]), 0)

subcategoria
Lesionados                               17
Hospitales afectados                      2
Escuelas afectadas                       40
Iglesias afectadas                       14
Edificios gubernamentales afectados      17
Hoteles afectados                         7
Puentes peatonales afectados              6
Albergues activos                         0
Personas albergadas                       0
Bolsas de alimentos del IMA            9693
Comunidades atendidas por el IMA        448

Bolsas de alimentos del IMA por comunidad atendida: 21.6
Correcto: no quedan albergues activos en Coclé


**Respuesta:** en Coclé hay 40 escuelas y 2 hospitales afectados. Ya no quedan albergues activos, y el IMA entregó 9,693 bolsas de alimentos en 448 comunidades, unas 22 por comunidad.

### 6.5 ¿Cuánto duraron las réplicas?

El Instituto de Geociencias de la Universidad de Panamá localizó unas 1,690 réplicas hasta el 20 de noviembre, ya menos de 10 por día. La gente siguió durmiendo afuera durante semanas.

**Lo aprendido:** frecuencia diaria a partir de una serie acumulada con `diff` ([unidad 38](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u38-series-temporales.ipynb)).

In [10]:
rep = t["replicas"].assign(fecha=lambda x: pd.to_datetime(x["fecha"])).set_index("fecha")["replicas_acumuladas"]
por_dia = (rep.diff() / rep.index.to_series().diff().dt.days).round(1)
print(pd.DataFrame({"acumuladas": rep, "réplicas por día en el tramo": por_dia}))

fig, ax = figura(10, 4.8)
ax.plot(rep.index, rep.values, color=AZUL, linewidth=2.5, marker="o", markersize=7)
for f, v in rep.items():
    ax.annotate(f"{v:,}", (f, v), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9, color=TINTA2)
ax.set_ylim(0, 1900)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{pd.Timestamp(plt.matplotlib.dates.num2date(x)).day}-"
                                               f"{['oct', 'nov'][pd.Timestamp(plt.matplotlib.dates.num2date(x)).month - 10]}"))
dias = (rep.index[-1] - pd.Timestamp("2026-10-09")).days
estilo(ax, f"Unas {rep.iloc[-1]:,} réplicas en {dias} días; el ritmo bajó de {por_dia.iloc[1]:.0f} a {por_dia.iloc[-1]:.0f} por día",
       "IGC-UP, según la prensa (Panamá América, Infobae, Destino Panamá, Prensa Latina, Crítica, La Verdad, La Prensa). Réplicas por día: cálculo propio.",
       ylabel="Réplicas acumuladas")
guardar(fig, "pulso-06-replicas")
plt.show()

            acumuladas  réplicas por día en el tramo
fecha                                               
2026-10-10         135                           NaN
2026-10-16         746                         101.8
2026-11-02        1271                          30.9
2026-11-06        1452                          45.2
2026-11-09        1483                          10.3
2026-11-17        1656                          21.6
2026-11-20        1690                          11.3


/tmp/ipykernel_1022084/96570017.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.6 ¿Cómo va la recuperación?

Un mes y medio después del sismo, estos son los datos más recientes de la reconstrucción y la atención social. Algunas cifras son de Coclé y otras de las cuatro provincias más afectadas.

**Lo aprendido:** ordenar y filtrar un DataFrame ([unidad 37](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u37-ordenacion-de-datos.ipynb)).

In [11]:
hoy = t["situacion"].sort_values("fecha", ascending=False)
cocle = hoy[hoy["area"].str.contains("Coclé|Penonomé")]
pd.set_option("display.max_colwidth", 60)
print(cocle[["fecha", "indicador", "valor", "unidad", "tipo_fuente"]].to_string(index=False))
proy = t["poblacion_proy"]
pen = proy[proy["area"] == "Penonomé (distrito)"].set_index("fecha")["poblacion"]
print()
print("Población del distrito de Penonomé:", pen.to_dict())
verificar("la proyección de 2024 coincide con la que ya usaba el estudio", int(pen.loc[2024]), 111457)

     fecha                                 indicador     valor          unidad           tipo_fuente
2026-11-23  Días sin cortes de luz por falta de pago     60.00            días               oficial
2026-11-17       Estudiantes sin clases presenciales 193000.00     estudiantes prensa (cita oficial)
2026-11-09      Viviendas en construcción del Miviot     67.00       viviendas               oficial
2026-11-09 Viviendas contratadas para reconstrucción    439.00       viviendas prensa (cita oficial)
2026-11-09            Monto de contratos de vivienda     12.68 millones de B/. prensa (cita oficial)
2026-11-06     Fichas sociales de emergencia (FIBES)   1890.00          fichas               oficial

Población del distrito de Penonomé: {2023: 104326, 2024: 111457, 2025: 112470, 2026: 113409}
Correcto: la proyección de 2024 coincide con la que ya usaba el estudio


**Respuesta:** la reconstrucción avanza lento. En Coclé hay 439 viviendas contratadas y 67 en obra, y 1,890 familias tienen ficha social de emergencia del MIDES. En las cuatro provincias más afectadas, unos 193,000 estudiantes seguían sin clases presenciales a mediados de noviembre. El distrito sigue creciendo: 113,409 habitantes proyectados para 2026.

## 7. ¿Dónde ayudar primero?

Este es el resultado que se vende: un **índice de prioridad** de 0 a 100 para cada corregimiento, que combina:

- **Pobreza** (50 %): pobreza multidimensional por corregimiento 2023.
- **Carencias de la vivienda** (30 %): promedio de viviendas con piso de tierra, sin agua potable y sin luz, escalado al corregimiento con más carencias.
- **Daños reportados** (20 %): si hubo al menos un registro de daños en la compilación de prensa (SINAPROC no publica la evaluación por corregimiento). Se usa sí o no, y no la cantidad, porque la prensa cubre mucho más la cabecera y contar registros premiaría la visibilidad y no la necesidad.

**Lo aprendido:** funciones con parámetros por defecto (los pesos), ordenamiento por fusión para el ranking ([unidad 28](../../01-python/semana-04/clase-1-algoritmos-de-ordenamiento/u28-ordenamiento-por-fusion.ipynb)) y búsqueda binaria para ubicar un umbral ([unidad 25](../../01-python/semana-03/clase-2-busqueda-binaria-tablas-hash/u25-busqueda-binaria.ipynb)).

In [12]:
corr["prioridad"] = pulso.indice_prioridad(corr)
ranking, _ = fusion(corr.to_dict("records"), clave=lambda c: -c["prioridad"])
tabla = pd.DataFrame(ranking)[["corregimiento", "prioridad", "ipm", "carencias_pct", "registros_danos", "pob_2023"]]
tabla.index = range(1, len(tabla) + 1)
print(tabla.to_string())

# ¿Cuántos corregimientos superan 50 puntos? Búsqueda binaria sobre los puntajes ordenados de menor a mayor
puntajes = sorted(corr["prioridad"])
pos = next(i for i, p in enumerate(puntajes) if p >= 50)
verificar("la búsqueda binaria encuentra el primer puntaje del umbral", bin_search(puntajes, puntajes[pos]), pos)
altos = tabla[tabla["prioridad"] >= 50]
print(f"Corregimientos con prioridad alta (50 o más): {len(altos)}, con {altos['pob_2023'].sum():,} habitantes")

                 corregimiento  prioridad  ipm  carencias_pct  registros_danos  pob_2023
1                         Tulú       85.4   76           33.3                1      2241
2                   San Miguel       83.2   81           27.5                1      2038
3                      Riecito       73.0   86           36.4                0      1315
4              Chiguirí Arriba       66.0   64           17.0                1      7185
5                    Río Indio       65.9   77           33.3                0      4026
6            Candelario Ovalle       60.4   75           27.8                0      3615
7                Boca de Tucué       58.8   71           28.3                0      2090
8                       Toabré       54.9   47           13.8                6      7440
9                      Pajonal       51.6   45           11.1                1      7678
10                   Las Minas       50.8   47            8.8                1      2662
11                  R

In [13]:
# Sensibilidad: ¿cambia el grupo de prioridad alta si todos los criterios pesan lo mismo?
iguales = pulso.indice_prioridad(corr, {"pobreza": 1, "carencias": 1, "danos": 1})
top_base = set(tabla.head(6)["corregimiento"])
top_igual = set(corr.assign(p=iguales).nlargest(6, "p")["corregimiento"])
print("Seis primeros con los pesos del proyecto:", sorted(top_base))
print("Seis primeros con pesos iguales:        ", sorted(top_igual))
nucleo = sorted(top_base & top_igual)
print("Coinciden:", len(nucleo), "de 6 ->", nucleo)
norte = set(corr.loc[corr["ipm"] >= 45, "corregimiento"])
verificar("con ambos pesos los seis primeros son del norte pobre (pobreza de 45 % o más)", (top_base | top_igual) <= norte, True)
verificar("Tulú, San Miguel, Riecito y Chiguirí Arriba encabezan con ambos pesos", nucleo,
          ["Chiguirí Arriba", "Riecito", "San Miguel", "Tulú"])

Seis primeros con los pesos del proyecto: ['Candelario Ovalle', 'Chiguirí Arriba', 'Riecito', 'Río Indio', 'San Miguel', 'Tulú']
Seis primeros con pesos iguales:         ['Chiguirí Arriba', 'Pajonal', 'Riecito', 'San Miguel', 'Toabré', 'Tulú']
Coinciden: 4 de 6 -> ['Chiguirí Arriba', 'Riecito', 'San Miguel', 'Tulú']
Correcto: con ambos pesos los seis primeros son del norte pobre (pobreza de 45 % o más)
Correcto: Tulú, San Miguel, Riecito y Chiguirí Arriba encabezan con ambos pesos


In [14]:
fig, ax = figura(10, 6.0)
orden = tabla.iloc[::-1]
colores = [NARANJA if p >= 50 else AZUL for p in orden["prioridad"]]
barras = ax.barh(orden["corregimiento"], orden["prioridad"], color=colores, height=0.7)
for b, (_, f) in zip(barras, orden.iterrows()):
    dano = "con daños reportados" if f["registros_danos"] > 0 else "sin registros de daños"
    ax.text(b.get_width() + 1, b.get_y() + b.get_height() / 2, f"{f['prioridad']:.0f}   pobreza {f['ipm']:.0f} %, {dano}",
            va="center", fontsize=8.5, color=TINTA2)
ax.set_xlim(0, 135)
ax.axvline(50, color="#c3c2b7", linewidth=1, linestyle="--")
estilo(ax, f"Dónde ayudar primero: {len(altos)} corregimientos del norte con prioridad alta",
       "Índice del proyecto: pobreza (IPM 2023) 50 %, carencias de vivienda (Censo 2023) 30 %, daños reportados 20 %.",
       xlabel="Índice de prioridad (0 a 100)", rejilla="x")
guardar(fig, "pulso-09-prioridad")
plt.show()

/tmp/ipykernel_1022084/157470836.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** diez corregimientos del norte, con 40,290 habitantes, tienen prioridad alta; Tulú, San Miguel, Riecito y Chiguirí Arriba encabezan con cualquier peso.

## 8. Resumen de hallazgos

Ordenados por su importancia para decidir:

1. **La ayuda tiene que ir primero al norte.** Diez corregimientos con 40,290 habitantes tienen prioridad alta; Tulú, San Miguel, Riecito y Chiguirí Arriba encabezan con cualquier combinación de pesos.
2. **Las personas más vulnerables necesitan seguimiento propio.** En Coclé, 91 personas con discapacidad y 9 mujeres embarazadas viven en viviendas con daño total o moderado.
3. **El daño fue sobre todo parcial, pero no menor.** De 2,531 viviendas evaluadas, 561 (22.2 %) quedaron con daño total o moderado; además hay 40 escuelas y 2 hospitales afectados.
4. **La prensa mira la cabecera y la necesidad está en otra parte.** La cabecera tiene 28 de los 43 registros de daños, pero pobreza de 16 %; en el norte hay pocos registros y pobreza de hasta 86 %.
5. **El sismo golpeó a un distrito que ya estaba frágil:** en las cinco carencias de vivienda medidas, Penonomé está peor que el país; la brecha mayor es internet (41.9 % frente a 30.3 %).

### Limitaciones y ética

- SINAPROC publica la evaluación de daños por provincia. No se puede repartir por corregimiento, así que el índice usa los registros de prensa solo como sí o no.
- Los registros de la compilación de prensa son registros, no unidades dañadas, y faltan las zonas aisladas.
- La EDAN cuenta viviendas evaluadas, no todas las de la provincia, y el bloque de otras afectaciones se actualiza a diario.
- El índice de prioridad depende de los pesos elegidos; por eso se probó con otros pesos y se informa qué parte del ranking es estable.
- Los datos son agregados, sin información personal. Señalar corregimientos pobres busca priorizar la ayuda, no estigmatizarlos.

**Recomendaciones:** validar el índice en campo con el municipio y SINAPROC, pedir a SINAPROC la evaluación de daños por corregimiento para cruzar daño y vulnerabilidad en el mismo nivel, y dar seguimiento médico a las personas con discapacidad y las embarazadas que viven en casas con daño grave.

[Proyecto](README.md) | [Estudio completo](documento/estudio-penonome.md) | [Anexo: técnicas por semana](notebooks/anexo-semanas/s01-listas-diccionarios-conjuntos.ipynb)